# Improved — retrieve-then-rerank with a local LLM

Baseline **+** a local Qwen3-8B that reranks the retrieved codes. The 8B model loads **4-bit** (~6 GB) so it fits a free Colab **T4**.

See `docs/03_improved.md`.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 1. Setup

Clone the `course` branch and install. Use a **GPU runtime** (Runtime → Change runtime type → T4 GPU).

In [ ]:
!git clone https://github.com/AIVIETNAM-AIO-DinhBao/ViClinicalIE_2.git medextract
%cd medextract
!pip -q install -r requirements.txt
!pip -q install -e .

# Fix lỗi transformers -> torchvision mismatch trên Colab.
# Repo này chỉ xử lý text, không cần torchvision/torchaudio/torchtext.
!pip -q uninstall -y torchvision torchaudio torchtext

## 2. Knowledge bases

The linking step needs the RxNorm + ICD-10 knowledge bases. Their source files are license-gated, so download them yourself (see `INSTALL.md`) and place them in `data/kb/raw/`, then build the indexes.

> If you already have prebuilt `data/kb/processed/*.parquet` + `*.faiss` (e.g. from Google Drive), copy them into `data/kb/processed/` and skip the build cell.

In [ ]:
# after placing the raw sources in data/kb/raw/ :
#!python -m medextract.kb.build_rxnorm
#!python -m medextract.kb.build_icd
#!python -m medextract.kb.index --device auto

!mkdir -p data/kb/processed
!unzip -o "/content/drive/Shareddrives/MY PLACE/kb_processed.zip" -d data/kb/processed
!ls -lh data/kb/processed

!mkdir -p data/input
!unzip -o "/content/drive/Shareddrives/MY PLACE/input_txt.zip" -d data/input
!ls data/input | head

!python -c "from pathlib import Path; req=['ICD10.faiss','ICD10_meta.parquet','icd_terms.parquet','RXNORM.faiss','RXNORM_meta.parquet','rxnorm_terms.parquet']; base=Path('data/kb/processed'); print('\n'.join(f'{x}: {(base/x).exists()}' for x in req)); print('num input txt:', len(list(Path('data/input').glob('*.txt'))))"

: 

## 3. Run the improved pipeline on the sample notes

First run downloads Qwen3-8B (a few minutes). `configs/improved.yaml` sets `load_in_4bit: true`.

In [ ]:
!python -c "from pathlib import Path; p=Path('configs/improved.yaml'); s=p.read_text(); s=s.replace('/mnt/pretrained_fm/Qwen_Qwen3-8B', 'Qwen/Qwen3-8B'); p.write_text(s); print(p.read_text())"

In [ ]:
!python run.py --config configs/improved.yaml --input data/sample_input --output out/demo_imp

## 4. Score baseline vs improved on the dev set

In [ ]:
!python run.py --config configs/improved.yaml --input data/dev/input --output out/dev_imp
!python score.py --pred out/dev_imp --gold data/dev

## 5. Build a submission zip

Point `--input` at the official test directory; `--zip` writes `out/sub/submission.zip`.

In [ ]:
!python run.py --config configs/improved.yaml --input data/input --output out/sub --zip
!ls -la out/sub/submission.zip